In [1]:
# Install dependencies
!pip install ultralytics roboflow -q

# Download dataset
from roboflow import Roboflow
rf = Roboflow(api_key="m12P1KJROPmY8XKmVIVO")
project = rf.workspace("brad-dwyer").project("pklot-1tros")
version = project.version(2)
dataset = version.download("yolov8-obb")   # change from yolov5-obb to yolov8-obb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 113.5 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to PKLot-2 in yolov8-obb:: 100%|██████████| 24844/24844 [00:03<00:00, 6956.38it/s] 


In [2]:
# Check what was downloaded
import os
print(os.listdir(dataset.location))


['valid', 'test', 'README.roboflow.txt', 'README.dataset.txt', 'data.yaml', 'train']


In [3]:
# Read and fix the data.yaml
# PKLot has 2 classes: empty + occupied — we merge them into 1 class "slot"
import yaml

yaml_path = dataset.location + "/data.yaml"
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

print("Original:", data)  # see what classes exist

# Merge to single class
data['nc'] = 1
data['names'] = ['slot']
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("Updated:", data)


Original: {'path': '../datasets/roboflow', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': {0: 'space-empty', 1: 'space-occupied'}}
Updated: {'path': '../datasets/roboflow', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': ['slot'], 'nc': 1}


In [6]:
import yaml

yaml_path = "/kaggle/working/PKLot-2/data.yaml"
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Fix the path to point to actual dataset location
data['path'] = '/kaggle/working/PKLot-2'
data['nc'] = 1
data['names'] = ['slot']

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print(data)


{'names': ['slot'], 'nc': 1, 'path': '/kaggle/working/PKLot-2', 'test': 'test/images', 'train': 'train/images', 'val': 'valid/images'}


In [7]:
# Relabel all annotation files: replace class 0 and 1 with just 0
import glob

label_files = glob.glob(dataset.location + "/**/*.txt", recursive=True)
for path in label_files:
    with open(path, 'r') as f:
        lines = f.readlines()
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if parts:
            parts[0] = '0'   # all slots become class 0
            new_lines.append(' '.join(parts) + '\n')
    with open(path, 'w') as f:
        f.writelines(new_lines)

print(f"Relabeled {len(label_files)} files")


Relabeled 12418 files


In [8]:
# Train
from ultralytics import YOLO

model = YOLO('yolov8n-obb.pt')  # nano = fastest, good starting point

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name='parking_slot_obb',
    patience=10,       # stop early if no improvement
    save=True,
)


Ultralytics 8.4.42 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/PKLot-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=parking_slot_obb-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

In [9]:
# See where the best model was saved
print(results.save_dir)
# Download it: runs/obb/parking_slot_obb/weights/best.pt


/kaggle/working/runs/obb/parking_slot_obb-2
